In [1]:
import torch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 假设我们有有效样本的查询隐藏态和良性样本
# 这里我们用随机数据来模拟（实际中你会有真实的隐藏态）

# 假设每个样本的维度是 d=512，样本数量是 N=1000
d = 512  # 隐藏层维度
N = 1000  # 样本数量

# 模拟有效样本（目标侧）和良性样本的隐藏态
# 注意：在实际情况中，这些应该来自你的模型输出
valid_samples = torch.randn(N, d)  # 有效样本的查询隐藏态
benign_samples = torch.randn(N, d)  # 良性样本的查询隐藏态

# 将样本转为numpy数组进行PCA处理
valid_samples_np = valid_samples.numpy()
benign_samples_np = benign_samples.numpy()

# 步骤 1: 构建目标语义子空间 U (通过有效样本的参考向量)
# 在有效样本中，假设我们已经计算了参考向量 r_star（例如，正确和错误答案之间的差异）
# 这里我们用随机数据模拟参考向量
r_star = torch.randn(d)  # 假设这是有效样本的参考纠偏向量

# 为了简化，假设参考向量 r_star 与每个有效样本的隐藏态的差异形成一个矩阵
# 用 PCA 对这些差异进行建模来得到子空间
valid_diffs = valid_samples - r_star  # 参考向量与样本之间的差异

# 标准化有效样本差异
scaler = StandardScaler()
valid_diffs_scaled = scaler.fit_transform(valid_diffs.numpy())

# 使用PCA提取目标语义子空间 U
pca = PCA(n_components=32)  # 选择前32个主成分
U = pca.fit_transform(valid_diffs_scaled)  # 得到目标子空间

# 步骤 2: 构建良性子空间 B (通过良性样本)
# 对良性样本应用 PCA，提取前b个主成分作为B
benign_samples_scaled = scaler.fit_transform(benign_samples.numpy())  # 标准化

# 使用 PCA 提取良性子空间 B
pca_benign = PCA(n_components=32)  # 选择前32个主成分
B = pca_benign.fit_transform(benign_samples_scaled)  # 得到良性子空间

# 打印结果
print("目标语义子空间 U 的形状:", U.shape)
print("良性子空间 B 的形状:", B.shape)



目标语义子空间 U 的形状: (1000, 32)
良性子空间 B 的形状: (1000, 32)


In [2]:
valid_samples.shape

torch.Size([1000, 512])

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 假设数据集已经准备好了，以下是模型训练框架

class SteeringAdapter(nn.Module):
    def __init__(self, input_dim,hidden_dim, k):
        super(SteeringAdapter, self).__init__()
        
        # 门控头（控制是否启用 steering）
        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        # 尺度头（调节沿 r_0 的强度）
        self.scale = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        # 混合头（在目标子空间 U 内做个性化修正）
        self.mix = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, k)  # k 是目标子空间的维度
        )

    def forward(self, h, r_0, U):
        # 计算门控值
        g = self.gate(h).squeeze(-1)  # gate output
        g = torch.clamp(g, 0.0, 1.0)
        
        # 计算尺度和混合头
        alpha = self.scale(h).squeeze(-1)
        beta = self.mix(h)

        # 计算最终的 steering 向量
        s_raw = alpha * r_0 + torch.matmul(beta, U.T)  # r_0 是全局方向，beta 在 U 中
        return g, alpha, beta, s_raw

In [ ]:
# d = 512  # 隐藏层维度
# N = 1000  # 样本数量

# # 模拟有效样本（目标侧）和良性样本的隐藏态
# valid_samples = torch.randn(N, d)  # 有效样本的查询隐藏态
# benign_samples = torch.randn(N, d)  # 良性样本的查询隐藏态

# # 构建目标语义子空间 U
# valid_samples_np = valid_samples.numpy()
# scaler = StandardScaler()
# valid_samples_scaled = scaler.fit_transform(valid_samples_np)
# pca = PCA(n_components=32)  # 选择前32个主成分
# U = pca.fit_transform(valid_samples_scaled)  # 得到目标子空间

In [ ]:
# # 构建良性子空间 B
# benign_samples_np = benign_samples.numpy()
# benign_samples_scaled = scaler.fit_transform(benign_samples_np)
# pca_benign = PCA(n_components=32)  # 选择前32个主成分
# B = pca_benign.fit_transform(benign_samples_scaled)  # 得到良性子空间

# # 模拟参考向量 r_star（有效样本的参考纠偏向量）
# r_star = torch.randn(d)

In [6]:
input_dim = 512  # 输入维度，问题的 hidden_state 维度
hidden_dim = 128  # 隐藏层维度
k = 32  # 目标子空间的维度（通过 PCA 得到）

# 模型初始化
model = SteeringAdapter(input_dim, hidden_dim, k)

# 假设数据的维度和参考向量的初始化
h = torch.randn(32, input_dim)  # 32 个问题的 hidden_state
r_star = torch.randn(input_dim)  # 参考向量
U = torch.randn(input_dim, k)  # 目标子空间（通过 PCA 得到）

# 优化器
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [7]:
# 损失函数
def loss_function(g, s_raw, Y_plus, Y_minus, alpha, beta):
    # 1. 门控损失：确保正确问题施加 steering，错误问题不施加
    gate_loss = F.binary_cross_entropy(g, torch.ones_like(g))  # 对正样本施加 1
    
    # 2. 差异优化损失：优化参考向量
    correct_diff = Y_plus - Y_minus  # 正确答案和不正确答案的差
    steering_loss = F.mse_loss(s_raw, correct_diff)  # 将 steering 向量与该差值进行比较

    # 3. 正则化
    reg_loss = F.mse_loss(alpha, torch.zeros_like(alpha)) + F.mse_loss(beta, torch.zeros_like(beta))
    
    return gate_loss + steering_loss + reg_loss

# 训练过程
num_epochs = 10  # 设置训练的 epoch 数量
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # 随机选择一个 batch 作为训练样本
    valid_batch = h  # 假设每次取32个有效样本
    correct_answer = torch.randn(32, input_dim)  # 假设的正确答案 hidden state
    incorrect_answer = torch.randn(32, input_dim)  # 假设的错误答案 hidden state

    # 训练门控头 g 和其他头
    optimizer.zero_grad()

    # 前向传播
    g, alpha, beta, s_raw = model(valid_batch, r_star, U)
    
    # 计算损失
    loss = loss_function(g, s_raw, correct_answer, incorrect_answer, alpha, beta)
    
    # 反向传播
    loss.backward()
    optimizer.step()
    
    # 打印训练信息
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

RuntimeError: The size of tensor a (32) must match the size of tensor b (512) at non-singleton dimension 0

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class SteeringModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, k):
        super(SteeringModel, self).__init__()
        
        # 门控头：决定是否施加 steering 向量
        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        # 尺度头：控制沿全局向量 r_0 的强度
        self.scale = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # 混合头：在目标子空间 U 内做个性化修正
        self.mix = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, k)  # k 是目标子空间的维度
        )

    def forward(self, h, r_0, U):
        # 计算门控值
        g = self.gate(h).squeeze(-1)  # gate output
        g = torch.clamp(g, 0.0, 1.0)  # 确保 gate 在 [0, 1] 之间
        
        # 计算尺度和混合头
        alpha = self.scale(h).squeeze(-1)
        beta = self.mix(h)
        
        # 计算最终的 steering 向量
        s_raw = alpha * r_0 + torch.matmul(beta, U.T)  # r_0 是全局方向，beta 在 U 中
        return g, alpha, beta, s_raw

# 假设我们有输入的数据
input_dim = 512  # 输入维度，问题的 hidden_state 维度
hidden_dim = 128  # 隐藏层维度
k = 32  # 目标子空间的维度（通过 PCA 得到）

# 模型初始化
model = SteeringModel(input_dim, hidden_dim, k)

# 假设数据的维度和参考向量的初始化
h = torch.randn(32, input_dim)  # 32 个问题的 hidden_state
r_star = torch.randn(input_dim)  # 参考向量
U = torch.randn(input_dim, k)  # 目标子空间（通过 PCA 得到）

# 优化器
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 损失函数
def loss_function(g, s_raw, Y_plus, Y_minus, alpha, beta):
    # 1. 门控损失：确保正确问题施加 steering，错误问题不施加
    gate_loss = F.binary_cross_entropy(g, torch.ones_like(g))  # 对正样本施加 1
    
    # 2. 差异优化损失：优化参考向量
    correct_diff = Y_plus - Y_minus  # 正确答案和不正确答案的差
    steering_loss = F.mse_loss(s_raw, correct_diff)  # 将 steering 向量与该差值进行比较

    # 3. 正则化
    reg_loss = F.mse_loss(alpha, torch.zeros_like(alpha)) + F.mse_loss(beta, torch.zeros_like(beta))
    
    return gate_loss + steering_loss + reg_loss

# 训练过程
num_epochs = 10  # 设置训练的 epoch 数量
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # 随机选择一个 batch 作为训练样本
    valid_batch = h  # 假设每次取32个有效样本
    correct_answer = torch.randn(32, input_dim)  # 假设的正确答案 hidden state
    incorrect_answer = torch.randn(32, input_dim)  # 假设的错误答案 hidden state

    # 训练门控头 g 和其他头
    optimizer.zero_grad()

    # 前向传播
    g, alpha, beta, s_raw = model(valid_batch, r_star, U)
    
    # 计算损失
    loss = loss_function(g, s_raw, correct_answer, incorrect_answer, alpha, beta)
    
    # 反向传播
    loss.backward()
    optimizer.step()
    
    # 打印训练信息
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


RuntimeError: The size of tensor a (32) must match the size of tensor b (512) at non-singleton dimension 0

In [9]:
r_star.shape

torch.Size([512])

In [17]:
a=[1,2]
a=torch.tensor(a)

In [12]:
torch.tensor(a)*r_star

RuntimeError: The size of tensor a (2) must match the size of tensor b (512) at non-singleton dimension 0

In [18]:
alpha_expanded = a.unsqueeze(-1)

In [19]:
alpha_expanded.shape

torch.Size([2, 1])

In [22]:
re=alpha_expanded*r_star

In [23]:
re.shape

torch.Size([2, 512])

In [26]:
re[0][:5],re[1,:5],r_star[:5]

(tensor([-0.4294, -0.4527,  0.0650,  0.9767, -0.6722]),
 tensor([-0.8588, -0.9053,  0.1301,  1.9534, -1.3444]),
 tensor([-0.4294, -0.4527,  0.0650,  0.9767, -0.6722]))

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class SteeringModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, k):
        super(SteeringModel, self).__init__()
        
        # 门控头：决定是否施加 steering 向量
        self.gate = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
        
        # 尺度头：控制沿全局向量 r_0 的强度
        self.scale = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # 混合头：在目标子空间 U 内做个性化修正
        self.mix = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, k)  # k 是目标子空间的维度
        )

    def forward(self, h, r_0, U):
        # 计算门控值
        g = self.gate(h).squeeze(-1)  # gate output
        g = torch.clamp(g, 0.0, 1.0)  # 确保 gate 在 [0, 1] 之间
        
        # 计算尺度和混合头
        alpha = self.scale(h).squeeze(-1)  # alpha 是沿 r_0 的强度
        beta = self.mix(h)  # beta 是在 U 子空间的个性化修正
        
        # 扩展 alpha 为与 r_0 相同的维度，逐元素乘法
        alpha_expanded = alpha.unsqueeze(-1) * r_0  # alpha 对应每个样本与 r_0 相乘
        
        # 计算最终的 steering 向量
        s_raw = alpha_expanded + torch.matmul(beta, U.T)  # r_0 是全局方向，beta 在 U 中
        return g, alpha, beta, s_raw

# 假设我们有输入的数据
input_dim = 512  # 输入维度，问题的 hidden_state 维度
hidden_dim = 128  # 隐藏层维度
k = 32  # 目标子空间的维度（通过 PCA 得到）

# 模型初始化
model = SteeringModel(input_dim, hidden_dim, k)

# 假设数据的维度和参考向量的初始化
h = torch.randn(32, input_dim)  # 32 个问题的 hidden_state
r_star = torch.randn(input_dim)  # 参考向量
U = torch.randn(input_dim, k)  # 目标子空间（通过 PCA 得到）

# 优化器
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 损失函数
def loss_function(g, s_raw, Y_plus, Y_minus, alpha, beta):
    # 1. 门控损失：确保正确问题施加 steering，错误问题不施加
    gate_loss = F.binary_cross_entropy(g, torch.ones_like(g))  # 对正样本施加 1
    
    # 2. 差异优化损失：优化参考向量
    correct_diff = Y_plus - Y_minus  # 正确答案和不正确答案的差
    steering_loss = F.mse_loss(s_raw, correct_diff)  # 将 steering 向量与该差值进行比较

    # 3. 正则化
    reg_loss = F.mse_loss(alpha, torch.zeros_like(alpha)) + F.mse_loss(beta, torch.zeros_like(beta))
    
    return gate_loss + steering_loss + reg_loss

# 训练过程
num_epochs = 10  # 设置训练的 epoch 数量
for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # 随机选择一个 batch 作为训练样本
    valid_batch = h  # 假设每次取32个有效样本
    correct_answer = torch.randn(32, input_dim)  # 假设的正确答案 hidden state
    incorrect_answer = torch.randn(32, input_dim)  # 假设的错误答案 hidden state

    # 训练门控头 g 和其他头
    optimizer.zero_grad()

    # 前向传播
    g, alpha, beta, s_raw = model(valid_batch, r_star, U)
    
    # 计算损失
    loss = loss_function(g, s_raw, correct_answer, incorrect_answer, alpha, beta)
    
    # 反向传播
    loss.backward()
    optimizer.step()
    
    # 打印训练信息
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")


Epoch [1/10], Loss: 4.8728
Epoch [2/10], Loss: 3.6285
Epoch [3/10], Loss: 2.9711
Epoch [4/10], Loss: 2.6988
Epoch [5/10], Loss: 2.5459
Epoch [6/10], Loss: 2.4862
Epoch [7/10], Loss: 2.4345
Epoch [8/10], Loss: 2.4180
Epoch [9/10], Loss: 2.3455
Epoch [10/10], Loss: 2.3082


In [1]:
import torch

In [4]:
aaa=torch.load("hc_all.pt")

In [5]:
aaa.shape

torch.Size([408, 3, 3584])

In [8]:
import torch
import dataset
from datasets import load_from_disk
def prepare_tqa_train_test_ds(tokenizer, ds_name, layers:[18,20,22]):
    ds = load_from_disk(ds_name)
    train_ds = ds["train"]
    test_ds = ds["test"] 

    def encode(example):
        return tokenizer(example["template_q"], return_tensors="pt", add_special_tokens=False)  
    
    attr_list = [f"y_win_layer{layer}" for layer in layers] + [f"y_lose_layer{layer}" for layer in layers]
    train_ds.set_format(type='torch', columns=attr_list)
    
    test_ds = test_ds.map(encode)
    test_ds.set_format(type='torch', columns=attr_list + ['question', 'template_q', 'input_ids', 'correct_answers', 'incorrect_answers'])
    
    return train_ds, test_ds

In [ ]:
model, tokenizer = load_model_and_tokenizer(args.model_name, device, torch_dtype)

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 8.7 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.4/584.4 kB 14.6 MB/s eta 0:00:0000:01
